# Simple TensorFlow.js Model Training (Efficient)
## Moisture Detection Image Classification

This notebook trains an image classification model and exports it directly to TensorFlow.js format.

**Setup**: Use GPU runtime in Google Colab for faster training.

## 1. Setup and Mount Drive

In [2]:
# Install required packages with dependency handling
import os
import random
import numpy as np
os.system('pip install --upgrade pip -q')
os.system('pip install tensorflowjs -q')

# Import packages
import tensorflow as tf
import tensorflowjs as tfjs
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set your data path - update this to your Google Drive folder
DATA_PATH = '/content/drive/MyDrive/Predicto_GPT_Taining_images/'  # Update this path

# Set max images per class for training (0 = use all images)
MAX_IMAGES_PER_CLASS = 100  # Change this to limit images per class

# Verify data path exists
if os.path.exists(DATA_PATH):
    print(f"✓ Data path found: {DATA_PATH}")
    classes = [d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d))]
    print(f"Classes found: {classes}")
    print(f"Max images per class: {MAX_IMAGES_PER_CLASS if MAX_IMAGES_PER_CLASS > 0 else 'All images'}")

    # Show actual image counts per class
    for class_name in classes:
        class_path = os.path.join(DATA_PATH, class_name)
        image_files = [f for f in os.listdir(class_path)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        total_images = len(image_files)
        will_use = min(total_images, MAX_IMAGES_PER_CLASS) if MAX_IMAGES_PER_CLASS > 0 else total_images
        print(f"  {class_name}: {total_images} total, will use {will_use}")

else:
    print(f"✗ Data path not found: {DATA_PATH}")
    print("Please update DATA_PATH to match your Google Drive folder structure")

Mounted at /content/drive
✓ Data path found: /content/drive/MyDrive/Predicto_GPT_Taining_images/
Classes found: ['50', '25', '200', '175', '350', '400', '300', '250', '450', '75', '0', '130', '100', 'Invalid']
Max images per class: 100
  50: 2000 total, will use 100
  25: 2000 total, will use 100
  200: 2002 total, will use 100
  175: 2000 total, will use 100
  350: 2000 total, will use 100
  400: 2000 total, will use 100
  300: 2000 total, will use 100
  250: 2000 total, will use 100
  450: 2000 total, will use 100
  75: 2000 total, will use 100
  0: 2000 total, will use 100
  130: 2001 total, will use 100
  100: 2000 total, will use 100
  Invalid: 2000 total, will use 100


## 2. Efficient Data Preparation (No File Copying)

In [4]:
class LimitedImageDataGenerator(ImageDataGenerator):
    def flow_from_directory_limited(self, directory, max_per_class=None, **kwargs):
        """Create a data generator with limited images per class without copying files"""

        if max_per_class is None or max_per_class <= 0:
            # Use standard flow_from_directory if no limit
            return super().flow_from_directory(directory, **kwargs)

        # Get all class directories
        class_dirs = [d for d in os.listdir(directory)
                     if os.path.isdir(os.path.join(directory, d))]

        # Create lists to store selected file paths and labels
        selected_files = []
        labels = []
        class_indices = {}

        for idx, class_name in enumerate(sorted(class_dirs)):
            class_indices[class_name] = idx
            class_path = os.path.join(directory, class_name)

            # Get all image files in this class
            image_files = [f for f in os.listdir(class_path)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

            # Randomly sample up to max_per_class images
            if len(image_files) > max_per_class:
                selected = random.sample(image_files, max_per_class)
            else:
                selected = image_files

            # Add selected files to our lists
            for img_file in selected:
                selected_files.append(os.path.join(class_path, img_file))
                labels.append(idx)

        print(f"Selected {len(selected_files)} images total")

        # Create a custom generator that reads from selected files
        return self._create_limited_generator(selected_files, labels, class_indices, **kwargs)

    def _create_limited_generator(self, file_paths, labels, class_indices,
                                 target_size=(224, 224), batch_size=32,
                                 class_mode='categorical', subset=None, **kwargs):
        """Create a generator from file paths list"""

        from tensorflow.keras.utils import to_categorical
        from tensorflow.keras.preprocessing import image

        # Convert labels to categorical if needed
        num_classes = len(class_indices)
        if class_mode == 'categorical':
            labels = to_categorical(labels, num_classes)

        # Split into train/validation if subset is specified
        if hasattr(self, 'validation_split') and self.validation_split:
            split_idx = int(len(file_paths) * (1 - self.validation_split))

            # Shuffle data
            combined = list(zip(file_paths, labels))
            random.shuffle(combined)
            file_paths, labels = zip(*combined)

            if subset == 'training':
                file_paths = file_paths[:split_idx]
                labels = labels[:split_idx]
            elif subset == 'validation':
                file_paths = file_paths[split_idx:]
                labels = labels[split_idx:]

        # Create a Keras Sequence class that properly inherits from tf.keras.utils.Sequence
        class LimitedSequence(tf.keras.utils.Sequence):
            def __init__(self, file_paths, labels, batch_size, target_size, preprocessor,
                        class_indices, num_classes):
                self.file_paths = list(file_paths)
                self.labels = list(labels)
                self.batch_size = batch_size
                self.target_size = target_size
                self.preprocessor = preprocessor
                self.class_indices = class_indices
                self.num_classes = num_classes
                self.samples = len(file_paths)
                self.on_epoch_end()

            def __len__(self):
                return int(np.ceil(len(self.file_paths) / self.batch_size))

            def __getitem__(self, idx):
                batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
                batch_x = []
                batch_y = []

                for i in batch_indices:
                    # Load and preprocess image
                    img = image.load_img(self.file_paths[i], target_size=self.target_size)
                    x = image.img_to_array(img)
                    x = self.preprocessor.standardize(x)  # Apply preprocessing

                    batch_x.append(x)
                    batch_y.append(self.labels[i])

                return np.array(batch_x), np.array(batch_y)

            def on_epoch_end(self):
                self.indices = np.arange(len(self.file_paths))
                np.random.shuffle(self.indices)

        return LimitedSequence(file_paths, labels, batch_size, target_size, self,
                             class_indices, num_classes)

In [5]:
# Create custom data generators with efficient limiting
train_datagen = LimitedImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.2
)

# Generate training data (no file copying!)
train_generator = train_datagen.flow_from_directory_limited(
    DATA_PATH,
    max_per_class=MAX_IMAGES_PER_CLASS,
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    subset='training'
)

# Generate validation data
validation_generator = train_datagen.flow_from_directory_limited(
    DATA_PATH,
    max_per_class=MAX_IMAGES_PER_CLASS,
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    subset='validation'
)

print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Number of classes: {train_generator.num_classes}")
print(f"Class indices: {train_generator.class_indices}")

Selected 1400 images total
Selected 1400 images total
Training samples: 1400
Validation samples: 1400
Number of classes: 14
Class indices: {'0': 0, '100': 1, '130': 2, '175': 3, '200': 4, '25': 5, '250': 6, '300': 7, '350': 8, '400': 9, '450': 10, '50': 11, '75': 12, 'Invalid': 13}


## 3. Model Creation

In [6]:
# Create model with MobileNetV2 transfer learning
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model initially
base_model.trainable = False

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model created and compiled successfully")
print(f"Total parameters: {model.count_params():,}")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model created and compiled successfully
Total parameters: 2,423,758


## 4. Training

In [7]:
# Training callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        patience=10,
        restore_best_weights=True,
        monitor='val_accuracy'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        patience=5,
        factor=0.5,
        monitor='val_accuracy'
    )
]

# Calculate steps per epoch
steps_per_epoch = train_generator.samples // train_generator.batch_size
validation_steps = validation_generator.samples // validation_generator.batch_size

# Phase 1: Train with frozen base model
print("Phase 1: Training with frozen base model...")
history1 = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=20,
    validation_data=validation_generator,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)

print(f"Phase 1 completed. Best validation accuracy: {max(history1.history['val_accuracy']):.4f}")

Phase 1: Training with frozen base model...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 1143s 13s/step - accuracy: 0.2053 - loss: 2.4836 - val_accuracy: 0.5869 - val_loss: 1.4032 - learning_rate: 0.0010
Epoch 2/20
 1/87 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.4375 - loss: 1.7417

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


87/87 ━━━━━━━━━━━━━━━━━━━━ 54s 632ms/step - accuracy: 0.4375 - loss: 1.7417 - val_accuracy: 0.5905 - val_loss: 1.3910 - learning_rate: 0.0010
Epoch 3/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 141s 1s/step - accuracy: 0.4665 - loss: 1.5129 - val_accuracy: 0.6588 - val_loss: 1.0598 - learning_rate: 0.0010
Epoch 4/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 49s 569ms/step - accuracy: 0.5625 - loss: 1.3850 - val_accuracy: 0.6573 - val_loss: 1.0544 - learning_rate: 0.0010
Epoch 5/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - accuracy: 0.5643 - loss: 1.2247 - val_accuracy: 0.7364 - val_loss: 0.8549 - learning_rate: 0.0010
Epoch 6/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 48s 563ms/step - accuracy: 0.5000 - loss: 1.0690 - val_accuracy: 0.7320 - val_loss: 0.8562 - learning_rate: 0.0010
Epoch 7/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 147s 2s/step - accuracy: 0.6048 - loss: 1.0919 - val_accuracy: 0.7342 - val_loss: 0.7838 - learning_rate: 0.0010
Epoch 8/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 53s 620ms/step - accuracy: 0.8125 - loss: 0.8697 - val_accura

In [8]:
# Phase 2: Fine-tuning with unfrozen base model
print("Phase 2: Fine-tuning with unfrozen base model...")

# Unfreeze base model
base_model.trainable = True

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Continue training
history2 = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=30,
    validation_data=validation_generator,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)

print(f"Phase 2 completed. Best validation accuracy: {max(history2.history['val_accuracy']):.4f}")
print("Training completed successfully!")

Phase 2: Fine-tuning with unfrozen base model...
Epoch 1/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 197s 2s/step - accuracy: 0.2580 - loss: 4.0792 - val_accuracy: 0.1415 - val_loss: 8.9873 - learning_rate: 1.0000e-04
Epoch 2/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 75s 877ms/step - accuracy: 0.6875 - loss: 1.0755 - val_accuracy: 0.1408 - val_loss: 8.9248 - learning_rate: 1.0000e-04
Epoch 3/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - accuracy: 0.5582 - loss: 1.1274 - val_accuracy: 0.1401 - val_loss: 10.7052 - learning_rate: 1.0000e-04
Epoch 4/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 78s 900ms/step - accuracy: 0.7500 - loss: 0.6639 - val_accuracy: 0.1401 - val_loss: 10.7815 - learning_rate: 1.0000e-04
Epoch 5/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 144s 1s/step - accuracy: 0.7120 - loss: 0.8067 - val_accuracy: 0.1336 - val_loss: 10.2941 - learning_rate: 1.0000e-04
Epoch 6/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 80s 933ms/step - accuracy: 0.6250 - loss: 1.5197 - val_accuracy: 0.1329 - val_loss: 10.2862 - learning_rate: 1.0000e-04
Epoch 7/30


## 5. Export to TensorFlow.js

In [10]:
# Create output directory
output_dir = '/content/moisture_detection_model'
os.makedirs(output_dir, exist_ok=True)

# Convert to TensorFlow.js format
print("Converting model to TensorFlow.js format...")
tfjs.converters.save_keras_model(
    model,
    output_dir,
  # quantization_bytes=2  # Reduce model size
)

print(f"✓ Model exported to: {output_dir}")
print("Files created:")
for file in os.listdir(output_dir):
    file_path = os.path.join(output_dir, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"  - {file} ({size_mb:.2f} MB)")

Converting model to TensorFlow.js format...
failed to lookup keras version from the file,
    this is likely a weight only file
✓ Model exported to: /content/moisture_detection_model
Files created:
  - model.json (0.15 MB)
  - group1-shard3of3.bin (1.25 MB)
  - group1-shard2of3.bin (4.00 MB)
  - group1-shard1of3.bin (4.00 MB)


## 6. Create Model Metadata

In [11]:
import json
from datetime import datetime

# Get final validation accuracy
final_accuracy = max(history2.history['val_accuracy']) if 'val_accuracy' in history2.history else max(history1.history['val_accuracy'])

# Create metadata
metadata = {
    "modelName": "moisture-detection-custom",
    "version": "1.0.0",
    "description": "Custom trained moisture detection model using MobileNetV2 transfer learning",
    "trainingDate": datetime.now().strftime("%Y-%m-%d"),
    "modelType": "Image Classification",
    "architecture": "MobileNetV2 + Custom Head",
    "inputShape": [224, 224, 3],
    "classes": list(train_generator.class_indices.keys()),
    "classIndices": train_generator.class_indices,
    "performance": {
        "validationAccuracy": float(final_accuracy),
        "trainingSamples": train_generator.samples,
        "validationSamples": validation_generator.samples
    },
    "preprocessing": {
        "rescale": "1/255",
        "targetSize": [224, 224]
    },
    "trainingConfig": {
        "maxImagesPerClass": MAX_IMAGES_PER_CLASS if MAX_IMAGES_PER_CLASS > 0 else "all",
        "efficientSampling": True
    }
}

# Save metadata
metadata_path = os.path.join(output_dir, 'metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print("✓ Metadata created")
print(f"Final validation accuracy: {final_accuracy:.4f}")
print(f"Classes: {list(train_generator.class_indices.keys())}")
print(f"Images per class limit: {MAX_IMAGES_PER_CLASS if MAX_IMAGES_PER_CLASS > 0 else 'No limit'}")
print("✓ Used efficient sampling (no file duplication)")

✓ Metadata created
Final validation accuracy: 0.1415
Classes: ['0', '100', '130', '175', '200', '25', '250', '300', '350', '400', '450', '50', '75', 'Invalid']
Images per class limit: 100
✓ Used efficient sampling (no file duplication)


## 7. Download Model Files

In [12]:
# Create zip file for download
import zipfile

zip_path = '/content/moisture_detection_model.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, output_dir)
            zipf.write(file_path, arcname)

print(f"✓ Model files zipped: {zip_path}")

# Download the zip file
from google.colab import files
files.download(zip_path)

print("\n🎉 Training completed successfully!")
print("\nNext steps:")
print("1. Extract the downloaded zip file")
print("2. Upload model.json and .bin files to your web server")
print("3. Update your application to use the new model")
print(f"4. Use the class indices: {train_generator.class_indices}")
print(f"\nModel trained efficiently with {MAX_IMAGES_PER_CLASS if MAX_IMAGES_PER_CLASS > 0 else 'all'} images per class (no file duplication)")

✓ Model files zipped: /content/moisture_detection_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Training completed successfully!

Next steps:
1. Extract the downloaded zip file
2. Upload model.json and .bin files to your web server
3. Update your application to use the new model
4. Use the class indices: {'0': 0, '100': 1, '130': 2, '175': 3, '200': 4, '25': 5, '250': 6, '300': 7, '350': 8, '400': 9, '450': 10, '50': 11, '75': 12, 'Invalid': 13}

Model trained efficiently with 100 images per class (no file duplication)
